In [2]:
from gams import transfer as gt
import pandas as pd

In [ ]:
# inputs
results_gdx_path = snakemake.input.modelresults
gen_comm_decomm_path = snakemake.input.gen_comm_decomm_path
store_comm_decomm_path = snakemake.input.store_comm_decomm_path

# extra params
model_year = int(snakemake.wildcards.model_year)

m = gt.Container(results_gdx_path, system_directory=snakemake.params.gamspath)


gen_comm_decomm_cap_full = pd.read_csv(gen_comm_decomm_path)
store_comm_decomm_cap_full = pd.read_csv(store_comm_decomm_path)


In [ ]:
# Get sets for for the different type of technologies
vre = list(m["vre"].records.g)
g = list(m["g"].records.uni)
s = list(m["s"].records.uni)

# Lifetime for decommission year
lifetime_tech_dict = snakemake.params.lifetime_tech

# Get current results for new installed cap
cap0_gen_new_r = m["var_new_vre_pcap_r"].records
cap0_gen_new_z = m["var_new_pcap_z"].records
cap0_store_new_z = m["var_new_store_pcap_z"].records

# Give the correct structure to dataframes
cap0_gen_new_r = (
    cap0_gen_new_r
    .rename(columns={"z":"zone","r":"region","level":"value","vre":"Technology"})
    .query("Technology in @vre") # only vre
    .assign(Year=model_year)
    [["Technology","zone","region","Year","value"]]
    .assign(type="commissioning_year")
    .query("value>1e-4")
)

cap0_gen_new_z = (
    cap0_gen_new_z
    .rename(columns={"z":"zone","level":"value","g":"Technology"})
    .query("Technology in @g") # not vre but g
    .query("Technology not in @vre")
    .assign(Year=model_year)
    .assign(region="")
    [["Technology","zone","region","Year","value"]]
    .assign(type="commissioning_year")
    .query("value>1e-4")
)

cap0_store_new_z = (
    cap0_store_new_z
    .rename(columns={"z":"zone","level":"value","s":"Technology"})
    .query("Technology in @s") # only s technologies (probably, it is not needed)
    .assign(Year=model_year)
    .assign(region="")
    [["Technology","zone","region","Year","value"]]
    .assign(type="commissioning_year")
    .query("value>1e-4")
)



In [ ]:
# Separate data into commissioned and decommissioned cap, adding correct year
cap0_gen_new_r = (
    pd.concat([
        cap0_gen_new_r,
        cap0_gen_new_r
        .assign(Year=(cap0_gen_new_r["Year"]+cap0_gen_new_r.Technology.map(lifetime_tech_dict).astype(int)))
        .assign(type="decommissioning_year")
    ])
    
)

cap0_gen_new_z = (
    pd.concat([
        cap0_gen_new_z,
        cap0_gen_new_z
        .assign(Year=(cap0_gen_new_z["Year"]+cap0_gen_new_z.Technology.map(lifetime_tech_dict).astype(int)))
        .assign(type="decommissioning_year"),
    ])
)

cap0_store_new_z = (
    pd.concat([
        cap0_store_new_z,
        cap0_store_new_z
        .assign(Year=(cap0_store_new_z["Year"]+cap0_store_new_z.Technology.map(lifetime_tech_dict).astype(int)))
        .assign(type="decommissioning_year"),
    ])
)

# Update information to the csv files
pd.concat([
    gen_comm_decomm_cap_full, # old
    cap0_gen_new_z, # new
    cap0_gen_new_r, # new
]).to_csv(gen_comm_decomm_path,index=False)


pd.concat([
    store_comm_decomm_cap_full,
    cap0_store_new_z,
]).to_csv(store_comm_decomm_path,index=False)